In [ ]:
import os
import re
import unicodedata
import pandas as pd
import numpy as np

### **Data Cleaning**

In [ ]:
input_folder = "Ai_Knowledge_Base"
output_folder = "cleaned_Ai_Knowledge_Base"

os.makedirs(output_folder, exist_ok=True)

duplicate_files = []

for file in os.listdir(input_folder):
    if file.endswith(".txt"):

        input_path = os.path.join(input_folder, file)
        output_path = os.path.join(output_folder, file)

        with open(input_path, "r", encoding="utf-8", errors="ignore") as f:
            text = f.read()
        text = re.sub(r"<.*?>", "", text) # HTML text
        text = unicodedata.normalize("NFKC", text) # Normlaized Unicode
        text = re.sub(r"[^A-Za-z0-9\s.,!?]", "", text) # Removed speacial Chracter
        text = re.sub(r"\s+", " ", text).strip() # removed White spaces
        text = re.sub(r"<.*?>",'',text)
        text = re.sub(r'https?://\S+|www\.\S+', '', text)
        text = re.sub(r"\S+@\S+", '', text)
        text = re.sub(r"@\w+", '', text)
        text = re.sub(r"#", '', text)
        text = re.sub(r"\d+", '', text)
        
        if not text:
            continue
        if text in duplicate_files:
            continue

        duplicate_files.append(text)
    
        with open(output_path, "w", encoding="utf-8") as f:
            f.write(text)

print("Cleaning completed!")
print("Cleaned files:", output_folder)

Cleaning completed!
Cleaned files: cleaned_Ai_Knowledge_Base


### **Metadata Generation**

In [ ]:
import csv
from datetime import datetime

folder = "cleaned_Ai_Knowledge_Base"

def get_category(text):

    text = text.lower()

    if "deep learning" in text:
        return "Deep Learning"
    elif "machine learning" in text:
        return "Machine Learning"
    elif "artificial intelligence" in text or " ai " in text:
        return "Artificial Intelligence"
    elif "neural network" in text:
        return "Neural Networks"
    elif "computer vision" in text:
        return "Computer Vision"
    elif "natural language processing" in text or "nlp" in text:
        return "Natural Language Processing"
    elif "data science" in text:
        return "Data Science"
    elif "python" in text:
        return "Python"
    elif "tensorflow" in text:
        return "TensorFlow"
    elif "pytorch" in text:
        return "PyTorch"
    else:
        return "Others"

with open("metadata.csv", "w", newline="", encoding="utf-8") as file:

    writer = csv.writer(file)

    writer.writerow([
        "Article Name",
        "URL",
        "Category",
        "Word Count",
        "Character Count",
        "Download Date"
    ])

    for filename in os.listdir(folder):
        if filename.endswith(".txt"):
            path = os.path.join(folder, filename)
            with open(path, "r", encoding="utf-8") as f:
                text = f.read()
            article_name = filename.replace(".txt", "")
            url = "Unknown"
            category = get_category(text)
            word_count = len(text.split())
            character_count = len(text)
            download_date = datetime.now().strftime("%Y-%m-%d")

            writer.writerow([
                article_name,
                url,
                category,
                word_count,
                character_count,
                download_date
            ])

print("metadata.csv!")


metadata.csv!


In [ ]:
df = pd.read_csv("metadata.csv")
display(df.head(10), df.isnull().sum(), df.duplicated().sum())

,Article Name,URL,Category,Word Count,Character Count,Download Date
0,A.I. Artificial Intelligence,Unknown,Artificial Intelligence,4160,25419,2026-08-15
1,Active learning (machine learning),Unknown,Machine Learning,1525,10046,2026-08-15
2,Adversarial machine learning,Unknown,Deep Learning,5255,34022,2026-08-15
3,AI art,Unknown,Deep Learning,3271,21945,2026-08-15
4,AI boom,Unknown,Machine Learning,2738,17606,2026-08-15
5,AI slop,Unknown,Artificial Intelligence,4922,30798,2026-08-15
6,AlexNet,Unknown,Deep Learning,1807,11478,2026-08-15
7,Applications of artificial intelligence,Unknown,Deep Learning,6885,47088,2026-08-15
8,Artificial general intelligence,Unknown,Deep Learning,5527,35030,2026-08-15
9,Artificial intelligence,Unknown,Deep Learning,13055,85404,2026-08-15


Article Name       0
URL                0
Category           0
Word Count         0
Character Count    0
Download Date      0
dtype: int64

np.int64(0)

## **Chunking, Encoding, Prompting and Retrieveing**

In [ ]:
import os
import faiss
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_ollama import ChatOllama

input_folder = "cleaned_Ai_Knowledge_Base"

documents = []
for filename in os.listdir(input_folder):
    if filename.endswith(".txt"):
        path = os.path.join(input_folder, filename)
        with open(path, "r", encoding="utf-8") as f:
            text = f.read()
        documents.append(Document(page_content=text, metadata={"source": filename}))

print(f"Total documents loaded: {len(documents)}")

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
splitter = SemanticChunker(embeddings)

semantic_docs = splitter.split_documents(documents)
print(f"Total chunks: {len(semantic_docs)}")

vector_db = FAISS.from_documents(semantic_docs, embeddings)
vector_db.save_local("faiss-db")

query = "what is Machine learning" 
results = vector_db.similarity_search(query, k=3)

llm = ChatOllama(model="llama3.2:1b", temperature=0)
context = "\n\n".join(doc.page_content for doc in results)

prompt = f"""
You are a helpful AI assistant.

Answer ONLY from the given context.

If the answer is not present, say:
"I couldn't find that information in the provided document."

Context:
{context}

Question:
{query}

Answer:
"""
response = llm.invoke(prompt)
print(response.content)

sources = sorted({doc.metadata.get("source", "Unknown") for doc in results})
print("Sources:", sources)

C:\Users\SAM\AppData\Local\Temp\ipykernel_7188\3017672839.py:3: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


Total documents loaded: 82


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Total chunks: 817
Machine learning is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without being explicitly programmed.
Sources: ['Artificial intelligence.txt', 'Machine learning.txt']


In [ ]:
import pickle

model_config = {
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "llm_model": "llama3.2:1b",
    "vector_db_path": "faiss-db",
    "top_k": 3
}

with open("rag_model.pkl", "wb") as f:
    pickle.dump(model_config, f)

print("rag_model.pkl saved successfully!")

rag_model.pkl saved successfully!
